In [1]:
import os
import pandas as pd
import numpy as np
import duckdb

: 

In [ ]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_demand AS

SELECT
    DATE_TRUNC(
        'hour',
        tpep_dropoff_datetime
    ) AS dropoff_hour,
    
    DOLocationID,
    COUNT(*) AS demand

FROM 'data/taxi.parquet'

WHERE
    tpep_dropoff_datetime IS NOT NULL
    AND DOLocationID IS NOT NULL
    
    AND YEAR(tpep_dropoff_datetime) = 2023

GROUP BY
    dropoff_hour,
    DOLocationID

ORDER BY
    dropoff_hour,
    DOLocationID
""")

In [ ]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_demand_features AS

SELECT
    dropoff_hour,
    DOLocationID,
    demand,

    MONTH(dropoff_hour) AS month,
    DAY(dropoff_hour) AS day,
    HOUR(dropoff_hour) AS hour,
    DAYOFWEEK(dropoff_hour) AS weekday,

    CASE
        WHEN DAYOFWEEK(dropoff_hour) IN (0, 6)
        THEN 1
        ELSE 0
    END AS is_weekend

FROM taxi_demand

ORDER BY
    dropoff_hour,
    DOLocationID
""")

In [ ]:
duckdb.sql("""
COPY taxi_demand_features
TO 'data/taxi_demand_processed_do.parquet'
(FORMAT PARQUET)
""")

In [ ]:
print(
    duckdb.sql("""
    SELECT COUNT(*) AS total_rows
    FROM 'data/taxi_demand_processed_do.parquet'
    """).fetchall()
)

print(
    duckdb.sql("""
    SELECT *
    FROM 'data/taxi_demand_processed_do.parquet'
    LIMIT 10
    """).df()
)

In [ ]:
print(
    duckdb.sql("""
    SELECT
        COUNT(*) AS n,
        MIN(demand) AS min_demand,
        MAX(demand) AS max_demand,
        AVG(demand) AS mean_demand,
        STDDEV(demand) AS std_demand,
        MEDIAN(demand) AS median_demand
    FROM 'data/taxi_demand_processed_do.parquet'
    """).df()
)